In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

food_path = os.path.join(path, 'Q1_data.csv')

df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:

df_food.head()

In [ ]:
# Task 3: Write your code here:

df_food.info()

In [ ]:
# Task 4: Write your code here:

df_food.describe()

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

df_clean = df_food.copy()


df_clean = df_clean.drop(columns = ['Order_ID'])

df_clean.info()

In [ ]:
# Task 2: Write your code here:

cols_drop = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']

df_clean = df_clean.dropna(subset = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])

df_clean.isnull().sum().sum()

df_clean.info()

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:

features = df_clean.columns.drop("Delivery_Time")

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])

df_clean.head()

In [ ]:
# Task 6: Write your code here:

# no need to check for imblance as its regression not categorial. only categorial data can be imbalanced

In [ ]:
# Task 1: Write your code here:

X = df_clean.drop("Delivery_Time", axis=1).astype(float)

y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error as sklearn_mae

rf = RandomForestRegressor(n_estimators=200)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_results = {'mae': []}

pred = []

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print(f"Training model...")
    print("-" * 30)


    # Train
    rf.fit(X_train, y_train)

    # Predict
    y_pred = rf.predict(X_test)

    pred.append(y_pred)

    # Calculate metrics
    mae = sklearn_mae(y_test, y_pred)

    # Store results
    all_results["mae"].append(mae)

print(f"  MAE for all folds:  {np.mean(all_results['mae']):.4f}")


In [ ]:
# Task 1: Write your code here:

from sklearn.linear_model import Ridge, Lasso

# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = rf.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(pred, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Precited Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q


def bonus():

  from catboost import CatBoostRegressor
  from sklearn.ensemble import RandomForestRegressor
  from sklearn.metrics import mean_absolute_error as sklearn_mae

  rf = RandomForestRegressor(n_estimators=200)

  cb = CatBoostRegressor(verbose=0)

  kf = KFold(n_splits=5, shuffle=True, random_state=42)

  all_results = {'mae': []}

  pred = []

  # Iterate through folds
  for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
      # indexing for each fold
      X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
      y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
      # print shapes
      print(f"Fold {fold}")
      print("  X_train shape:", X_train.shape)
      print("  X_test shape :", X_test.shape)
      print("  y_train shape:", y_train.shape)
      print("  y_test shape :", y_test.shape)
      print(f"Training model...")
      print("-" * 30)


      # Train
      rf.fit(X_train, y_train)
      cb.fit(X_train, y_train)

      # Predict
      rf_y_pred = rf.predict(X_test)
      cb_y_pred = cb.predict(X_test)

      avg_pred = (rf_y_pred + cb_y_pred)/2

      # Calculate metrics
      mae = sklearn_mae(y_test, avg_pred)

      # Store results
      all_results["mae"].append(mae)

  print(f"  MAE for all folds:  {np.mean(all_results['mae']):.4f}")

bonus()
